<a href="https://colab.research.google.com/github/jasminewongjiaxuan/marketing-campaign-response-prediction/blob/main/marketing_campaign_prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import files
uploaded = files.upload()

Import libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

Load dataset

In [ ]:
df = pd.read_csv("marketing_campaign.csv", sep="\t")
df.head()

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
print(df.shape)

Handling Missing Values

In [ ]:
print("\nMissing values before handling:")
print(df.isnull().sum())

df["Income"] = df["Income"].fillna(0)

print("\nMissing values after handling:")
print(df.isnull().sum())

Removing Identifier Column

In [ ]:
if "ID" in df.columns:
    df.drop("ID", axis=1, inplace=True)

print("\nColumns after dropping ID:")
print(df.columns)

Data Type and Datetime Conversion

In [ ]:
print("\nData types before conversion:")
print(df.dtypes)

df["Dt_Customer"] = pd.to_datetime(df["Dt_Customer"], format="%d-%m-%Y", errors="coerce")

print("\nData types after conversion:")
print(df.dtypes)


Feature Engineering

In [ ]:
df["Total_Campaign_Accepted"] = (
    df["AcceptedCmp1"] +
    df["AcceptedCmp2"] +
    df["AcceptedCmp3"] +
    df["AcceptedCmp4"] +
    df["AcceptedCmp5"]
)

df["Total_Children"] = df["Kidhome"] + df["Teenhome"]

df["Total_Spend"] = (
    df["MntWines"] +
    df["MntFruits"] +
    df["MntMeatProducts"] +
    df["MntFishProducts"] +
    df["MntSweetProducts"] +
    df["MntGoldProds"]
)

df["Age"] = 2025 - df["Year_Birth"]

reference_date = df["Dt_Customer"].max()
df["Customer_Days"] = (reference_date - df["Dt_Customer"]).dt.days

print("\nDataset after feature engineering:")
print(df.head())

Removing Irrelevant Variables

In [ ]:
irrelevant_cols = ["Z_CostContact", "Z_Revenue", "Complain"]
df.drop(columns=[col for col in irrelevant_cols if col in df.columns], inplace=True)

print("\nColumns after dropping irrelevant variables:")
print(df.columns)

Handling Duplicate Records

In [ ]:
print("\nDuplicate rows before removing:", df.duplicated().sum())

df = df.drop_duplicates()

print("Duplicate rows after removing:", df.duplicated().sum())
print("Dataset shape after removing duplicates:", df.shape)

Removing Redundant Variables

In [ ]:
redundant_cols = [
    "AcceptedCmp1", "AcceptedCmp2", "AcceptedCmp3", "AcceptedCmp4", "AcceptedCmp5",
    "Kidhome", "Teenhome",
    "MntWines", "MntFruits", "MntMeatProducts",
    "MntFishProducts", "MntSweetProducts", "MntGoldProds",
    "Year_Birth", "Dt_Customer"
]

df.drop(columns=[col for col in redundant_cols if col in df.columns], inplace=True)

print("\nColumns after dropping redundant variables:")
print(df.columns)
print("Dataset shape:", df.shape)

Feature Selection

In [ ]:
import os
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
import pandas as pd
import matplotlib.pyplot as plt

df_fs = df.copy()

if "Response_Label" in df_fs.columns:
    df_fs = df_fs.drop("Response_Label", axis=1)

label_encoders = {}
categorical_cols_fs = df_fs.select_dtypes(include=["object"]).columns

for col in categorical_cols_fs:
    le = LabelEncoder()
    df_fs[col] = le.fit_transform(df_fs[col])
    label_encoders[col] = le


X_fs = df_fs.drop("Response", axis=1)
y_fs = df_fs["Response"]

rf_fs = RandomForestClassifier(random_state=42)
rf_fs.fit(X_fs, y_fs)

feature_importance = pd.DataFrame({
    "Feature": X_fs.columns,
    "Importance": rf_fs.feature_importances_
}).sort_values(by="Importance", ascending=False).reset_index(drop=True)

print("\nFeature importance:")
print(feature_importance)

top10_features = feature_importance.head(10)

colors = [
    "#4B0082",  # purple
    "#1f77b4",  # blue
    "#2ca02c",  # green
    "#ffbf00",  # yellow
    "#ff7f0e",  # orange
    "#d62728",  # red
    "#9467bd",  # violet
    "#17becf",  # cyan
    "#8c564b",  # brown
    "#e377c2"   # pink
]

plt.figure(figsize=(10, 6))

bars = plt.barh(
    top10_features["Feature"],
    top10_features["Importance"],
    color=colors[:len(top10_features)]
)

plt.gca().invert_yaxis()

for bar in bars:
    width = bar.get_width()
    plt.text(
        width + 0.001,
        bar.get_y() + bar.get_height() / 2,
        f"{width:.3f}",
        va="center",
        fontsize=9
    )

plt.title("Top 10 Feature Importance (Random Forest)")
plt.xlabel("Importance Score")
plt.ylabel("Feature")
plt.tight_layout()
os.makedirs("chapter4_figures", exist_ok=True)
plt.savefig("chapter4_figures/Figure_4_10_Top_10_Feature_Importance.png", dpi=300)
plt.show()

Encoding Categorical Variables

In [ ]:
X = df.drop("Response", axis=1)
y = df["Response"]

numerical_cols = X.select_dtypes(include=["int64", "float64"]).columns.tolist()
categorical_cols = X.select_dtypes(include=["object"]).columns.tolist()

print("\nNumerical columns:", numerical_cols)
print("Categorical columns:", categorical_cols)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numerical_cols),
        ("cat", OneHotEncoder(drop="first", handle_unknown="ignore"), categorical_cols)
    ]
)

print("\nPreprocessor:")
print(preprocessor)

Data Splitting

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.3,
    stratify=y,
    random_state=42
)

print("\nTrain and test shapes:")
print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

Feature Scaling

In [ ]:
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

print("\nProcessed train and test shapes:")
print("X_train_processed:", X_train_processed.shape)
print("X_test_processed:", X_test_processed.shape)

Logistic Regression

In [ ]:
lr_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", LogisticRegression(max_iter=1000, random_state=42))
])

lr_pipeline.fit(X_train, y_train)

y_pred_lr = lr_pipeline.predict(X_test)
y_prob_lr = lr_pipeline.predict_proba(X_test)[:, 1]

print("\nLogistic Regression Results")
print("ROC-AUC:", roc_auc_score(y_test, y_prob_lr))
print("Accuracy:", accuracy_score(y_test, y_pred_lr))
print("Precision:", precision_score(y_test, y_pred_lr, zero_division=0))
print("Recall:", recall_score(y_test, y_pred_lr, zero_division=0))
print("F1-Score:", f1_score(y_test, y_pred_lr, zero_division=0))

Naive Bayes

In [ ]:
X_train_nb = preprocessor.fit_transform(X_train)
X_test_nb = preprocessor.transform(X_test)

if hasattr(X_train_nb, "toarray"):
    X_train_nb = X_train_nb.toarray()
    X_test_nb = X_test_nb.toarray()

nb_model = GaussianNB()
nb_model.fit(X_train_nb, y_train)

y_pred_nb = nb_model.predict(X_test_nb)
y_prob_nb = nb_model.predict_proba(X_test_nb)[:, 1]

print("\nNaive Bayes Results")
print("ROC-AUC:", roc_auc_score(y_test, y_prob_nb))
print("Accuracy:", accuracy_score(y_test, y_pred_nb))
print("Precision:", precision_score(y_test, y_pred_nb, zero_division=0))
print("Recall:", recall_score(y_test, y_pred_nb, zero_division=0))
print("F1-Score:", f1_score(y_test, y_pred_nb, zero_division=0))

K-Nearest Neighbors (KNN)

In [ ]:
knn_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", KNeighborsClassifier())
])

knn_pipeline.fit(X_train, y_train)

y_pred_knn = knn_pipeline.predict(X_test)
y_prob_knn = knn_pipeline.predict_proba(X_test)[:, 1]

print("\nKNN Results")
print("ROC-AUC:", roc_auc_score(y_test, y_prob_knn))
print("Accuracy:", accuracy_score(y_test, y_pred_knn))
print("Precision:", precision_score(y_test, y_pred_knn, zero_division=0))
print("Recall:", recall_score(y_test, y_pred_knn, zero_division=0))
print("F1-Score:", f1_score(y_test, y_pred_knn, zero_division=0))

Decision Tree

In [ ]:
dt_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", DecisionTreeClassifier(random_state=42))
])

dt_pipeline.fit(X_train, y_train)

y_pred_dt = dt_pipeline.predict(X_test)
y_prob_dt = dt_pipeline.predict_proba(X_test)[:, 1]

print("\nDecision Tree Results")
print("ROC-AUC:", roc_auc_score(y_test, y_prob_dt))
print("Accuracy:", accuracy_score(y_test, y_pred_dt))
print("Precision:", precision_score(y_test, y_pred_dt, zero_division=0))
print("Recall:", recall_score(y_test, y_pred_dt, zero_division=0))
print("F1-Score:", f1_score(y_test, y_pred_dt, zero_division=0))

Random Forest

In [ ]:
rf_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", RandomForestClassifier(n_estimators=100, random_state=42))
])

rf_pipeline.fit(X_train, y_train)

y_pred_rf = rf_pipeline.predict(X_test)
y_prob_rf = rf_pipeline.predict_proba(X_test)[:, 1]

print("\nRandom Forest Results")
print("ROC-AUC:", roc_auc_score(y_test, y_prob_rf))
print("Accuracy:", accuracy_score(y_test, y_pred_rf))
print("Precision:", precision_score(y_test, y_pred_rf, zero_division=0))
print("Recall:", recall_score(y_test, y_pred_rf, zero_division=0))
print("F1-Score:", f1_score(y_test, y_pred_rf, zero_division=0))

XGBoost

In [ ]:
xgb_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", XGBClassifier(
        use_label_encoder=False,
        eval_metric="logloss",
        n_estimators=100,
        learning_rate=0.1,
        max_depth=3,
        random_state=42
    ))
])

xgb_pipeline.fit(X_train, y_train)

y_pred_xgb = xgb_pipeline.predict(X_test)
y_prob_xgb = xgb_pipeline.predict_proba(X_test)[:, 1]

print("\nXGBoost Results")
print("ROC-AUC:", roc_auc_score(y_test, y_prob_xgb))
print("Accuracy:", accuracy_score(y_test, y_pred_xgb))
print("Precision:", precision_score(y_test, y_pred_xgb, zero_division=0))
print("Recall:", recall_score(y_test, y_pred_xgb, zero_division=0))
print("F1-Score:", f1_score(y_test, y_pred_xgb, zero_division=0))

Evaluation Metrics

In [ ]:
results = pd.DataFrame({
    "Model": [
        "Logistic Regression",
        "Naive Bayes",
        "KNN",
        "Decision Tree",
        "Random Forest",
        "XGBoost"
    ],
    "ROC-AUC": [
        roc_auc_score(y_test, y_prob_lr),
        roc_auc_score(y_test, y_prob_nb),
        roc_auc_score(y_test, y_prob_knn),
        roc_auc_score(y_test, y_prob_dt),
        roc_auc_score(y_test, y_prob_rf),
        roc_auc_score(y_test, y_prob_xgb)
    ],
    "Accuracy": [
        accuracy_score(y_test, y_pred_lr),
        accuracy_score(y_test, y_pred_nb),
        accuracy_score(y_test, y_pred_knn),
        accuracy_score(y_test, y_pred_dt),
        accuracy_score(y_test, y_pred_rf),
        accuracy_score(y_test, y_pred_xgb)
    ],
    "Precision": [
        precision_score(y_test, y_pred_lr, zero_division=0),
        precision_score(y_test, y_pred_nb, zero_division=0),
        precision_score(y_test, y_pred_knn, zero_division=0),
        precision_score(y_test, y_pred_dt, zero_division=0),
        precision_score(y_test, y_pred_rf, zero_division=0),
        precision_score(y_test, y_pred_xgb, zero_division=0)
    ],
    "Recall": [
        recall_score(y_test, y_pred_lr, zero_division=0),
        recall_score(y_test, y_pred_nb, zero_division=0),
        recall_score(y_test, y_pred_knn, zero_division=0),
        recall_score(y_test, y_pred_dt, zero_division=0),
        recall_score(y_test, y_pred_rf, zero_division=0),
        recall_score(y_test, y_pred_xgb, zero_division=0)
    ],
    "F1-Score": [
        f1_score(y_test, y_pred_lr, zero_division=0),
        f1_score(y_test, y_pred_nb, zero_division=0),
        f1_score(y_test, y_pred_knn, zero_division=0),
        f1_score(y_test, y_pred_dt, zero_division=0),
        f1_score(y_test, y_pred_rf, zero_division=0),
        f1_score(y_test, y_pred_xgb, zero_division=0)
    ]
})

results = results.sort_values(by="ROC-AUC", ascending=False).reset_index(drop=True)

print("\nFinal Model Comparison Results:")
print(results)

In [ ]:
df["Response"].unique()

Distribution of Customer Response

In [ ]:
df["Response_Label"] = df["Response"].map({0: "No Response", 1: "Response"})
response_counts = df["Response_Label"].value_counts()

response_counts = response_counts[["No Response", "Response"]]

plt.figure(figsize=(6, 4))
bars = plt.bar(
    response_counts.index,
    response_counts.values,
    color=["#d62728", "#2ca02c"]
)

for bar in bars:
    height = bar.get_height()
    plt.text(
        bar.get_x() + bar.get_width() / 2,
        height + 5,
        f"{int(height)}",
        ha="center",
        va="bottom"
    )

plt.title("Distribution of Customer Response")
plt.xlabel("Customer Response")
plt.ylabel("Number of Customers")
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig("chapter4_figures/Figure_4_1_Distribution_of_Customer_Response.png", dpi=300)
plt.show()

Demographic and Behavioural Patterns by Customer Response

Boxplot Comparison of Recency, Total Spending, Income, and Age by Customer Response

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(3, 2, figsize=(12, 12))

# 1. Recency
recency_data = [
    df[df["Response"] == 0]["Recency"].dropna(),
    df[df["Response"] == 1]["Recency"].dropna()
]

axes[0, 0].boxplot(
    recency_data,
    labels=["No Response", "Response"]
)

axes[0, 0].set_title("Recency by Customer Response")
axes[0, 0].set_xlabel("Customer Response")
axes[0, 0].set_ylabel("Recency")


# 2. Customer_Days
customer_days_data = [
    df[df["Response"] == 0]["Customer_Days"].dropna(),
    df[df["Response"] == 1]["Customer_Days"].dropna()
]

axes[0, 1].boxplot(
    customer_days_data,
    labels=["No Response", "Response"]
)

axes[0, 1].set_title("Customer Days by Customer Response")
axes[0, 1].set_xlabel("Customer Response")
axes[0, 1].set_ylabel("Customer Days")


# 3. Total_Spend
spend_data = [
    df[df["Response"] == 0]["Total_Spend"].dropna(),
    df[df["Response"] == 1]["Total_Spend"].dropna()
]

axes[1, 0].boxplot(
    spend_data,
    labels=["No Response", "Response"]
)

axes[1, 0].set_title("Total Spending by Customer Response")
axes[1, 0].set_xlabel("Customer Response")
axes[1, 0].set_ylabel("Total Spending")


# 4. Income
income_data = [
    df[df["Response"] == 0]["Income"].dropna(),
    df[df["Response"] == 1]["Income"].dropna()
]

axes[1, 1].boxplot(
    income_data,
    labels=["No Response", "Response"]
)

axes[1, 1].set_title("Income Distribution by Customer Response")
axes[1, 1].set_xlabel("Customer Response")
axes[1, 1].set_ylabel("Income")


# 5. Age
age_data = [
    df[df["Response"] == 0]["Age"].dropna(),
    df[df["Response"] == 1]["Age"].dropna()
]

axes[2, 0].boxplot(
    age_data,
    labels=["No Response", "Response"]
)

axes[2, 0].set_title("Age Distribution by Customer Response")
axes[2, 0].set_xlabel("Customer Response")
axes[2, 0].set_ylabel("Age")


# Remove empty subplot
fig.delaxes(axes[2, 1])

plt.tight_layout()

plt.savefig(
    "chapter4_figures/Figure_4_2_Boxplot_Comparison_by_Customer_Response.png",
    dpi=300
)

plt.show()

Purchase Channels by Customer Response

Purchase Channels by Customer Response

In [ ]:
purchase_cols = ["NumWebPurchases", "NumCatalogPurchases", "NumStorePurchases"]

purchase_channel_mean = df.groupby("Response_Label")[purchase_cols].mean()

purchase_channel_mean = purchase_channel_mean.loc[["No Response", "Response"]]

ax = purchase_channel_mean.plot(
    kind="bar",
    figsize=(8, 5),
    color=["#1f77b4", "#9467bd", "#2ca02c"]
)

plt.title("Purchase Channels by Customer Response")
plt.xlabel("Customer Response")
plt.ylabel("Average Number of Purchases")
plt.xticks(rotation=0)
plt.legend(
    ["NumWebPurchases", "NumCatalogPurchases", "NumStorePurchases"],
    title="Purchase Channel"
)

for container in ax.containers:
    ax.bar_label(container, fmt="%.2f", padding=3)

plt.tight_layout()
plt.savefig("chapter4_figures/Figure_4_8_Purchase_Channels_by_Customer_Response.png", dpi=300)
plt.show()

Total Campaigns Accepted by Customer Response

In [ ]:
campaign_response = pd.crosstab(df["Total_Campaign_Accepted"], df["Response_Label"])

campaign_response = campaign_response[["No Response", "Response"]]

ax = campaign_response.plot(
    kind="bar",
    figsize=(8, 5),
    color=["#d62728", "#2ca02c"]
)

plt.title("Total Campaigns Accepted by Customer Response")
plt.xlabel("Total Campaigns Accepted")
plt.ylabel("Number of Customers")
plt.xticks(rotation=0)
plt.legend(title="Response")

for container in ax.containers:
    ax.bar_label(container, fmt="%d", padding=3)

plt.tight_layout()
plt.savefig("chapter4_figures/Figure_4_9_Total_Campaigns_Accepted_by_Customer_Response.png", dpi=300)
plt.show()

ROC-AUC Comparison by Model

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.metrics import roc_auc_score

roc_auc_data = {
    "Model": [
        "Logistic Regression",
        "Naive Bayes",
        "K-Nearest Neighbors",
        "Decision Tree",
        "Random Forest",
        "XGBoost"
    ],
    "ROC-AUC": [
        roc_auc_score(y_test, y_prob_lr),
        roc_auc_score(y_test, y_prob_nb),
        roc_auc_score(y_test, y_prob_knn),
        roc_auc_score(y_test, y_prob_dt),
        roc_auc_score(y_test, y_prob_rf),
        roc_auc_score(y_test, y_prob_xgb)
    ]
}

roc_auc_df = pd.DataFrame(roc_auc_data)

roc_auc_df = roc_auc_df.sort_values(by="ROC-AUC", ascending=False)

colors = [
    "#d62728",  # red
    "#ff7f0e",  # orange
    "#ffbf00",  # yellow
    "#2ca02c",  # green
    "#1f77b4",  # blue
    "#4B0082"   # purple
]

plt.figure(figsize=(9, 5))

bars = plt.bar(
    roc_auc_df["Model"],
    roc_auc_df["ROC-AUC"],
    color=colors[:len(roc_auc_df)]
)

for bar in bars:
    height = bar.get_height()
    plt.text(
        bar.get_x() + bar.get_width() / 2,
        height + 0.01,
        f"{height:.3f}",
        ha="center",
        va="bottom",
        fontsize=10
    )

plt.title("ROC-AUC Scores by Model")
plt.xlabel("Model")
plt.ylabel("ROC-AUC")
plt.ylim(0, 1.05)
plt.xticks(rotation=45, ha="right")
plt.tight_layout()

plt.savefig("Figure_4_ROC_AUC_Comparison.png", dpi=300, bbox_inches="tight")

plt.show()

Overall Evaluation Metrics Comparison

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score, accuracy_score, precision_score, recall_score, f1_score

results_data = {
    "Model": [
        "Logistic Regression",
        "Naive Bayes",
        "K-Nearest Neighbors",
        "Decision Tree",
        "Random Forest",
        "XGBoost"
    ],
    "ROC-AUC": [
        roc_auc_score(y_test, y_prob_lr),
        roc_auc_score(y_test, y_prob_nb),
        roc_auc_score(y_test, y_prob_knn),
        roc_auc_score(y_test, y_prob_dt),
        roc_auc_score(y_test, y_prob_rf),
        roc_auc_score(y_test, y_prob_xgb)
    ],
    "Accuracy": [
        accuracy_score(y_test, y_pred_lr),
        accuracy_score(y_test, y_pred_nb),
        accuracy_score(y_test, y_pred_knn),
        accuracy_score(y_test, y_pred_dt),
        accuracy_score(y_test, y_pred_rf),
        accuracy_score(y_test, y_pred_xgb)
    ],
    "Precision": [
        precision_score(y_test, y_pred_lr, zero_division=0),
        precision_score(y_test, y_pred_nb, zero_division=0),
        precision_score(y_test, y_pred_knn, zero_division=0),
        precision_score(y_test, y_pred_dt, zero_division=0),
        precision_score(y_test, y_pred_rf, zero_division=0),
        precision_score(y_test, y_pred_xgb, zero_division=0)
    ],
    "Recall": [
        recall_score(y_test, y_pred_lr, zero_division=0),
        recall_score(y_test, y_pred_nb, zero_division=0),
        recall_score(y_test, y_pred_knn, zero_division=0),
        recall_score(y_test, y_pred_dt, zero_division=0),
        recall_score(y_test, y_pred_rf, zero_division=0),
        recall_score(y_test, y_pred_xgb, zero_division=0)
    ],
    "F1-Score": [
        f1_score(y_test, y_pred_lr, zero_division=0),
        f1_score(y_test, y_pred_nb, zero_division=0),
        f1_score(y_test, y_pred_knn, zero_division=0),
        f1_score(y_test, y_pred_dt, zero_division=0),
        f1_score(y_test, y_pred_rf, zero_division=0),
        f1_score(y_test, y_pred_xgb, zero_division=0)
    ]
}

results_df = pd.DataFrame(results_data)

results_df = results_df.sort_values(by="ROC-AUC", ascending=False).reset_index(drop=True)

results_df_display = results_df.copy()
results_df_display[["ROC-AUC", "Accuracy", "Precision", "Recall", "F1-Score"]] = results_df_display[
    ["ROC-AUC", "Accuracy", "Precision", "Recall", "F1-Score"]
].round(6)

print("Final Model Comparison Results:")
print(results_df_display)

fig, ax = plt.subplots(figsize=(11, 3.5))
ax.axis("off")

table = ax.table(
    cellText=results_df_display.values,
    colLabels=results_df_display.columns,
    cellLoc="center",
    loc="center"
)

table.auto_set_font_size(False)
table.set_fontsize(10)
table.scale(1.2, 1.5)

for (row, col), cell in table.get_celld().items():
    cell.set_edgecolor("black")
    cell.set_linewidth(1)
    if row == 0:
        cell.set_text_props(weight="bold")
        cell.set_facecolor("#f2f2f2")

plt.tight_layout()
plt.savefig("chapter4_figures/Table_4_1_Overall_Evaluation_Metrics_Comparison.png", dpi=300, bbox_inches="tight")
plt.show()